## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats
from sklearn.preprocessing import LabelEncoder

In [3]:
!pip install scikit-learn

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB 145.2 kB/s eta 0:00:57
   ---------------------------------------- 0.0/8.3 MB 145.2 kB/s eta 0:00:57
   ---------------------------------------- 0.0/8.3 MB 131.3 kB/s eta 0:01:03
   ---------------------------------------- 0.1/8.3 MB 172.4 kB/s eta 0:00:48
   ---------------------------------------- 0.1/8.3 MB 172.4 kB/s eta 0:00:48
   ---------------------------------------- 0.1/8.3 MB 199.7 kB/s eta 0:00:42
   ---------------------------------------- 0.1/8.3 MB 194.1 kB/s eta 0:00:43
   ---------------------------------------- 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Load the placement dataset into a Pandas Dataframe.

In [3]:
df=pd.read_csv("data.csv")
df.info()
df.shape
df.head()
df.tail()
df.sample(5)
df.describe()
df.loc[0]
df.iloc[0]
df[0:2]
df["degree_p"]
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

## Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [4]:
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,91.00,Commerce,58.00,Sci&Tech,55.0,Mkt&HR,58.80,270000.0
1,2,M,78.33,Science,77.48,Sci&Tech,86.5,Mkt&Fin,66.28,200000.0
2,3,M,NaN,Arts,64.00,Comm&Mgmt,75.0,Mkt&Fin,57.80,250000.0
3,4,M,52.00,Science,NaN,Sci&Tech,66.0,Mkt&HR,59.43,NaN
4,5,M,73.60,Commerce,73.30,Comm&Mgmt,96.8,Mkt&Fin,55.50,425000.0


In [5]:
df.tail()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
210,211,M,82.0,Commerce,77.6,Comm&Mgmt,91.0,Mkt&Fin,74.49,400000.0
211,212,M,60.0,Science,72.0,Sci&Tech,74.0,Mkt&Fin,53.62,275000.0
212,213,M,67.0,Commerce,73.0,Comm&Mgmt,59.0,Mkt&Fin,69.72,295000.0
213,214,F,66.0,Commerce,58.0,Comm&Mgmt,70.0,Mkt&HR,60.23,204000.0
214,215,M,58.0,Science,53.0,Comm&Mgmt,89.0,Mkt&HR,60.22,NaN


In [6]:
df.shape

(215, 10)

In [7]:
df.columns

Index(['sl_no', 'gender', 'hsc_p', 'hsc_s', 'degree_p', 'degree_t', 'etest_p',
       'specialisation', 'mba_p', 'salary'],
      dtype='str')

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


In [9]:
df.describe()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary
count,215.000000,210.000000,213.000000,211.000000,214.000000,148.000000
mean,108.000000,66.503000,66.420610,72.093649,62.254813,288655.405405
std,62.209324,10.904205,7.322786,13.340195,5.836962,93457.452420
min,1.000000,37.000000,50.000000,50.000000,51.210000,200000.000000
25%,54.500000,61.000000,61.000000,60.000000,57.922500,240000.000000
50%,108.000000,65.000000,66.000000,70.000000,61.950000,265000.000000
75%,161.500000,73.000000,72.000000,84.000000,66.187500,300000.000000
max,215.000000,97.700000,91.000000,98.000000,77.890000,940000.000000


In [10]:
df.isnull().sum()



sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

In [11]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

Missing values in each column:
sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

Total missing values:
79


## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [12]:
df.dropna(subset=["salary"], inplace=True)
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

In [12]:
df["hsc_p"].fillna(df["hsc_p"].mean(), inplace=True)
df["degree_p"].fillna(df["degree_p"].mean(), inplace=True)
df["etest_p"].fillna(df["etest_p"].mean(), inplace=True)

df.isnull().sum()

C:\Users\HP\AppData\Local\Temp\ipykernel_22352\339640050.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["hsc_p"].fillna(df["hsc_p"].mean(), inplace=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_22352\339640050.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

In [13]:
print(df.isnull().sum())

sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64


## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [14]:
c=["hsc_p","degree_p","etest_p","salary"]
s1=StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.646702,Sci&Tech,-1.319511,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.342287,Sci&Tech,0.971699,Mkt&Fin,66.28,-0.951839
2,3,M,1.533482e-15,Arts,-0.726068,Comm&Mgmt,0.135226,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.700913,Comm&Mgmt,1.720888,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.419191,Sci&Tech,-0.446669,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [15]:
s2=MinMaxScaler()
df[c]=s2.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [24]:
from scipy import stats

columns_to_check = ["salary"]

# Calculate Z-scores
z_score = stats.zscore(df[columns_to_check])

# Find outliers
u = z_score > 3
l = z_score < -3

# Combine upper and lower outlier conditions
indx = (u | l).any(axis=1)

# Remove outliers
clean_df = df[~indx]

# Check data
df.info()
clean_df.info()


<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    int64  
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    int64  
 4   degree_p        147 non-null    float64
 5   degree_t        148 non-null    int64  
 6   etest_p         146 non-null    float64
 7   specialisation  148 non-null    int64  
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(5)
memory usage: 12.7 KB
<class 'pandas.DataFrame'>
Index: 145 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           145 non-null    int64  
 1   gender          145 non-null    int64  
 2   hsc_p           145 non-null    float64
 3   h

### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [21]:

l1=df["salary"].quantile(0.05)
u1=df["salary"].quantile(0.95)
df_capped=df.copy()
df_capped["salary"]=df_capped["salary"].clip(l1,u1)
df_capped.info()
df.head()

<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        147 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         146 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.7 KB


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




In [14]:
df_encoded = df.copy()

le = LabelEncoder()

df_encoded["gender_encoded"] = le.fit_transform(df_encoded["gender"])

print("Original and encoded values:")
print(df_encoded[["gender", "gender_encoded"]].head())

print("\nMapping:")
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

Original and encoded values:
  gender  gender_encoded
0      M               1
1      M               1
2      M               1
3      M               1
4      M               1

Mapping:
{'F': np.int64(0), 'M': np.int64(1)}


## Convert categorical variables into numerical format using one hot encoder

[![Picture2.png](https://i.postimg.cc/gcZ0Hv1J/Picture2.png)](https://postimg.cc/HjTHp7YD)

In [15]:
one_hot = pd.get_dummies(
    df,
    columns=["gender", "hsc_s", "degree_t", "specialisation"],
    dtype=int
)

one_hot.head()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary,gender_F,gender_M,hsc_s_Arts,hsc_s_Commerce,hsc_s_Science,degree_t_Comm&Mgmt,degree_t_Others,degree_t_Sci&Tech,specialisation_Mkt&Fin,specialisation_Mkt&HR
0,1,91.00,58.00,55.0,58.80,270000.0,0,1,0,1,0,0,0,1,0,1
1,2,78.33,77.48,86.5,66.28,200000.0,0,1,0,0,1,0,0,1,1,0
2,3,NaN,64.00,75.0,57.80,250000.0,0,1,1,0,0,1,0,0,1,0
3,4,52.00,NaN,66.0,59.43,NaN,0,1,0,0,1,0,0,1,0,1
4,5,73.60,73.30,96.8,55.50,425000.0,0,1,0,1,0,1,0,0,1,0


In [18]:
one_hot.to_csv("Pre.csv", index=False)

print("Preprocessed dataset saved as Pre.csv")

Preprocessed dataset saved as Pre.csv


In [19]:

one_hot.head()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary,gender_F,gender_M,hsc_s_Arts,hsc_s_Commerce,hsc_s_Science,degree_t_Comm&Mgmt,degree_t_Others,degree_t_Sci&Tech,specialisation_Mkt&Fin,specialisation_Mkt&HR
0,1,91.00,58.00,55.0,58.80,270000.0,0,1,0,1,0,0,0,1,0,1
1,2,78.33,77.48,86.5,66.28,200000.0,0,1,0,0,1,0,0,1,1,0
2,3,NaN,64.00,75.0,57.80,250000.0,0,1,1,0,0,1,0,0,1,0
3,4,52.00,NaN,66.0,59.43,NaN,0,1,0,0,1,0,0,1,0,1
4,5,73.60,73.30,96.8,55.50,425000.0,0,1,0,1,0,1,0,0,1,0


# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.

In [20]:
# ============================================
# AUTOMOBILE DATASET
# Data Cleaning and Transformation
# ============================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import zscore


# --------------------------------------------
# STEP 1: LOAD AND EXPLORE DATASET
# --------------------------------------------

auto = pd.read_csv("Automobile.csv")

print("First 10 rows:")
display(auto.head(10))

print("\nShape of dataset:")
print(auto.shape)

print("\nColumn names:")
print(auto.columns.tolist())

print("\nData types:")
print(auto.dtypes)

print("\nDescriptive statistics:")
display(auto.describe(include="all"))

print("\nNumerical columns:")
print(auto.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical columns:")
print(auto.select_dtypes(exclude=np.number).columns.tolist())


# Display unique values of categorical columns
print("\nUnique values:")
for col in auto.select_dtypes(exclude=np.number).columns:
    print(col, ":", auto[col].unique())


# --------------------------------------------
# STEP 2: DATA CLEANING
# --------------------------------------------

# 3. Check missing values

print("\nMissing values:")
print(auto.isnull().sum())

missing_count = auto.isnull().sum()
missing_percentage = (missing_count / len(auto)) * 100

missing_table = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

print("\nMissing value summary:")
display(missing_table)


# --------------------------------------------
# 4. HANDLE MISSING VALUES
# --------------------------------------------

# Replace ? with NaN
auto = auto.replace("?", np.nan)

# Convert horsepower to numeric
auto["horsepower"] = pd.to_numeric(
    auto["horsepower"],
    errors="coerce"
)

# Fill numerical missing values with median
numeric_columns = auto.select_dtypes(include=np.number).columns

for col in numeric_columns:
    auto[col] = auto[col].fillna(auto[col].median())


# Fill categorical missing values with mode
categorical_columns = auto.select_dtypes(exclude=np.number).columns

for col in categorical_columns:
    if auto[col].isnull().any():
        auto[col] = auto[col].fillna(auto[col].mode()[0])


print("\nMissing values after cleaning:")
print(auto.isnull().sum())


# --------------------------------------------
# 5. REMOVE DUPLICATES
# --------------------------------------------

print("\nNumber of duplicate rows:")
print(auto.duplicated().sum())

auto = auto.drop_duplicates()

print("\nShape after removing duplicates:")
print(auto.shape)

print("\nDuplicate rows after removal:")
print(auto.duplicated().sum())


# --------------------------------------------
# 6. CLEAN HORSEPOWER COLUMN
# --------------------------------------------

print("\nHorsepower data type:")
print(auto["horsepower"].dtype)

print("\nHorsepower:")
print(auto["horsepower"].head())

# Ensure numeric
auto["horsepower"] = pd.to_numeric(
    auto["horsepower"],
    errors="coerce"
)

# Fill missing horsepower
auto["horsepower"] = auto["horsepower"].fillna(
    auto["horsepower"].median()
)

print("\nMissing horsepower:")
print(auto["horsepower"].isnull().sum())


# --------------------------------------------
# STEP 3: DATA TRANSFORMATION
# --------------------------------------------

# 7. Transform origin column

print("\nOriginal origin values:")
print(auto["origin"].unique())

origin_map = {
    1: "usa",
    2: "europe",
    3: "japan"
}

auto["origin"] = auto["origin"].map(origin_map)

print("\nTransformed origin:")
print(auto["origin"].unique())


# --------------------------------------------
# 8. CREATE weight_kg
# --------------------------------------------

auto["weight_kg"] = auto["weight"] * 0.453592

print("\nWeight conversion:")
display(auto[["weight", "weight_kg"]].head())


# --------------------------------------------
# 9. CREATE mpg_category
# --------------------------------------------

def mpg_category(mpg):
    if mpg < 20:
        return "Low"
    elif mpg < 30:
        return "Medium"
    else:
        return "High"


auto["mpg_category"] = auto["mpg"].apply(mpg_category)

print("\nMPG categories:")
print(auto[["mpg", "mpg_category"]].head())


# --------------------------------------------
# 10. CREATE vehicle_age
# --------------------------------------------

auto["vehicle_age"] = 2026 - auto["model_year"]

print("\nVehicle age:")
display(auto[["model_year", "vehicle_age"]].head())


# --------------------------------------------
# 11. RENAME COLUMNS
# --------------------------------------------

auto.rename(columns={
    "mpg": "miles_per_gallon",
    "horsepower": "hp",
    "weight": "weight_lbs",
    "model_year": "year"
}, inplace=True)

print("\nRenamed columns:")
print(auto.columns.tolist())


# --------------------------------------------
# 12. FILTER DATA
# --------------------------------------------

print("\nVehicles with MPG > 30:")
display(auto[auto["miles_per_gallon"] > 30])

print("\nVehicles with horsepower > 150:")
display(auto[auto["hp"] > 150])

print("\nVehicles with cylinders >= 6:")
display(auto[auto["cylinders"] >= 6])

print("\nVehicles manufactured after 1980:")
display(auto[auto["year"] > 80])

print("\nVehicles originating from USA:")
display(auto[auto["origin"] == "usa"])


# --------------------------------------------
# STEP 4: ENCODING CATEGORICAL DATA
# --------------------------------------------

# 13. Label Encoding

label_encoder = LabelEncoder()

auto["origin_encoded"] = label_encoder.fit_transform(
    auto["origin"]
)

print("\nLabel encoding:")
display(auto[["origin", "origin_encoded"]].head())

print("\nCategory to label mapping:")

origin_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

print(origin_mapping)


# --------------------------------------------
# 14. ONE-HOT ENCODING
# --------------------------------------------

one_hot_auto = pd.get_dummies(
    auto,
    columns=["origin"],
    dtype=int
)

print("\nOne-hot encoded dataset:")
display(one_hot_auto.head())


# --------------------------------------------
# STEP 5: OUTLIER DETECTION
# --------------------------------------------

numeric_cols = auto.select_dtypes(
    include=np.number
).columns

z_scores = np.abs(
    zscore(auto[numeric_cols])
)

outlier_counts = (z_scores > 3).sum(axis=0)

print("\nOutlier count in each numerical column:")
print(
    pd.Series(
        outlier_counts,
        index=numeric_cols
    )
)


# Display rows containing outliers

outlier_rows = auto[
    (z_scores > 3).any(axis=1)
]

print("\nRows containing outliers:")
display(outlier_rows)


# --------------------------------------------
# STEP 6: STANDARDIZATION
# --------------------------------------------

# Select numerical features
scaling_columns = [
    "cylinders",
    "displacement",
    "horsepower" if "horsepower" in auto.columns else "hp",
    "weight_lbs",
    "acceleration",
    "year"
]

standard_scaler = StandardScaler()

standardized_data = auto.copy()

standardized_data[scaling_columns] = standard_scaler.fit_transform(
    standardized_data[scaling_columns]
)

print("\nStandardized values:")
display(standardized_data[scaling_columns].head())


print("\nMean after standardization:")
print(standardized_data[scaling_columns].mean())

print("\nStandard deviation after standardization:")
print(standardized_data[scaling_columns].std())


# --------------------------------------------
# STEP 7: MIN-MAX NORMALIZATION
# --------------------------------------------

minmax_scaler = MinMaxScaler()

normalized_data = auto.copy()

normalized_data[scaling_columns] = minmax_scaler.fit_transform(
    normalized_data[scaling_columns]
)

print("\nNormalized values:")
display(normalized_data[scaling_columns].head())

print("\nMinimum values:")
print(normalized_data[scaling_columns].min())

print("\nMaximum values:")
print(normalized_data[scaling_columns].max())


# --------------------------------------------
# STEP 8: CREATE X AND Y
# --------------------------------------------

# Y = target variable
Y = auto["miles_per_gallon"]

# X = independent variables
X = auto.drop(
    columns=[
        "miles_per_gallon",
        "mpg_category",
        "origin"
    ],
    errors="ignore"
)

# Convert any remaining categorical columns to numeric
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

print("\nShape of X:")
print(X.shape)

print("\nShape of Y:")
print(Y.shape)


# Save X and Y
XY = X.copy()
XY["target_mpg"] = Y

XY.to_csv(
    "automobile_X_Y.csv",
    index=False
)

print("\nautomobile_X_Y.csv saved successfully.")


# Load CSV again
loaded_XY = pd.read_csv(
    "automobile_X_Y.csv"
)

print("\nFirst 5 rows of automobile_X_Y.csv:")
display(loaded_XY.head())


# --------------------------------------------
# STEP 9: FINAL PREPROCESSED DATASET
# --------------------------------------------

final_dataset = one_hot_auto.copy()

# Check missing values
print("\nMissing values in final dataset:")
print(final_dataset.isnull().sum().sum())

print("\nFinal dataset:")
display(final_dataset.head())

# Save final dataset
final_dataset.to_csv(
    "automobile_preprocessed.csv",
    index=False
)

print(
    "\nautomobile_preprocessed.csv saved successfully."
)

First 10 rows:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa
4,ford torino,17.0,NaN,302.0,140.0,3449.0,10.5,70,usa
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa
6,chevrolet impala,14.0,8.0,454.0,220.0,NaN,9.0,70,usa
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa
8,pontiac catalina,14.0,8.0,455.0,NaN,4425.0,10.0,70,usa
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa



Shape of dataset:
(398, 9)

Column names:
['name', 'mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']

Data types:
name                str
mpg             float64
cylinders       float64
displacement    float64
horsepower      float64
weight          float64
acceleration    float64
model_year        int64
origin              str
dtype: object

Descriptive statistics:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398,398.000000,395.000000,395.000000,386.000000,396.000000,395.000000,398.000000,398
unique,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,ford pinto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,usa
freq,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,249
mean,NaN,23.514573,5.445570,193.340506,104.316062,2965.025253,15.562278,76.010050,NaN
std,NaN,7.815984,1.696203,104.425993,38.086281,845.254458,2.750260,3.697627,NaN
min,NaN,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,NaN
25%,NaN,17.500000,4.000000,102.500000,75.250000,2222.250000,13.850000,73.000000,NaN
50%,NaN,23.000000,4.000000,146.000000,92.500000,2797.500000,15.500000,76.000000,NaN
75%,NaN,29.000000,8.000000,262.000000,125.000000,3581.750000,17.150000,79.000000,NaN



Numerical columns:
['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']

Categorical columns:
['name', 'origin']

Unique values:
name : <StringArray>
[ 'chevrolet chevelle malibu',          'buick skylark 320',
         'plymouth satellite',              'amc rebel sst',
                'ford torino',           'ford galaxie 500',
           'chevrolet impala',          'plymouth fury iii',
           'pontiac catalina',         'amc ambassador dpl',
 ...
 'chrysler lebaron medallion',             'ford granada l',
           'toyota celica gt',          'dodge charger 2.2',
           'chevrolet camaro',            'ford mustang gl',
                  'vw pickup',              'dodge rampage',
                'ford ranger',                 'chevy s-10']
Length: 305, dtype: str
origin : <StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str

Missing values:
name             0
mpg              0
cylinders        3
displacement     3
horsepo

...


Minimum values:
cylinders       0.0
displacement    0.0
hp              0.0
weight_lbs      0.0
acceleration    0.0
year            0.0
dtype: float64

Maximum values:
cylinders       1.0
displacement    1.0
hp              1.0
weight_lbs      1.0
acceleration    1.0
year            1.0
dtype: float64

Shape of X:
(398, 313)

Shape of Y:
(398,)

automobile_X_Y.csv saved successfully.

First 5 rows of automobile_X_Y.csv:


,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_encoded,name_amc ambassador dpl,...,name_volvo 244dl,name_volvo 245,name_volvo 264gl,name_volvo diesel,name_vw dasher (diesel),name_vw pickup,name_vw rabbit,name_vw rabbit c (diesel),name_vw rabbit custom,target_mpg
0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,1956,0,0,...,0,0,0,0,0,0,0,0,0,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,1956,0,0,...,0,0,0,0,0,0,0,0,0,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,1956,0,0,...,0,0,0,0,0,0,0,0,0,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,1956,0,0,...,0,0,0,0,0,0,0,0,0,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,1956,0,0,...,0,0,0,0,0,0,0,0,0,17.0



Missing values in final dataset:
0

Final dataset:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,mpg_category,vehicle_age,origin_encoded
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,Low,1956,0
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,Low,1956,0
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,Low,1956,0
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,Low,1956,0
4,ford torino,17.0,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,Low,1956,0



automobile_preprocessed.csv saved successfully.
